<a href="https://colab.research.google.com/github/vcasadei/Redes-Neurais-CESAR-School/blob/2026.01-A/5%20-%20Exemplos%20de%20Aplica%C3%A7%C3%A3o/5.6_preco_de_carros_usados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 5.6 — Preço de carros usados: limpeza de texto, one-hot e alvo em log

Mais um exemplo de **regressão**: prever o preço de um carro usado a partir do ano, da quilometragem, do motor, da marca, do combustível e do câmbio. Os dados vêm de anúncios de um site de carros usados da Índia, com preços em rúpias.

A rede e o laço de treino são os do notebook 5.5, que por sua vez seguem o 4.6. O que muda é a **preparação dos dados**, que neste conjunto é quase todo o trabalho:

- quase todas as colunas são texto (`'86,226 kms'`, `'1956 cc'`, `'10.03 Lakh'`) e precisam virar números;
- marca e combustível são **categorias**, e entram na rede como colunas 0/1 (*one-hot*);
- os preços vão de 35 mil a 19 milhões de rúpias, e por isso a rede aprende o **logaritmo** do preço.

## Objetivos

- Converter colunas de texto em números com as funções de texto do pandas (`.str`).
- Codificar variáveis categóricas com one-hot (`pd.get_dummies`) e entender por que não usar números arbitrários para categorias.
- Treinar com o alvo em escala logarítmica e voltar para rúpias na avaliação.
- Usar o modelo treinado para estimar o preço de um carro novo, aplicando a mesma preparação dos dados.

## Importação das bibliotecas

In [ ]:
import os
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(1111)
np.random.seed(1111)

device = "cuda" if torch.cuda.is_available() else "cpu"
print('device:', device)

## Dados

Conjunto [Old Car Price Prediction](https://www.kaggle.com/datasets/milanvaddoriya/old-car-price-prediction), do Kaggle: 5.512 anúncios. O endereço da API do Kaggle baixa o zip sem login. Se falhar, baixe pela página e coloque `car_price.csv` em `data/`.

In [ ]:
os.makedirs("data", exist_ok=True)
url = "https://www.kaggle.com/api/v1/datasets/download/milanvaddoriya/old-car-price-prediction"
zip_path = "data/old-car-price-prediction.zip"
if not os.path.exists(zip_path):
    urllib.request.urlretrieve(url, zip_path)
with zipfile.ZipFile(zip_path) as z:
    z.extractall("data")

df = pd.read_csv("data/car_price.csv")
print(df.shape)
df.head()

In [ ]:
df.info()

`df.info()` mostra que não há valores faltantes, mas só `manufacture` (ano de fabricação) é numérica (`int64`); as outras colunas são texto (`object`). `Unnamed: 0` é só o número da linha e será descartada.

### Do texto para números

Cada coluna tem um formato:

| coluna | exemplo | o que fazer |
|---|---|---|
| `kms_driven` | `'86,226 kms'` | tirar `' kms'` e a vírgula → 86226 |
| `engine` | `'1956 cc'` | tirar `' cc'` → 1956 (cilindrada do motor) |
| `Seats` | `'5 Seats'` | tirar `' Seats'` → 5 |
| `ownership` | `'1st Owner'` | primeiro caractere → 1 (quantos donos o carro já teve) |
| `transmission` | `'Manual'` / `'Automatic'` | 1 se automático, 0 se manual |
| `car_name` | `'Jeep Compass 2.0 ...'` | primeira palavra → marca (`'Jeep'`) |

O preço precisa de mais cuidado. Na Índia, 1 *lakh* = 100 mil e 1 *crore* = 10 milhões. A maioria dos preços vem em lakh, alguns em crore, e os mais baratos vêm sem unidade, em rúpias:

In [ ]:
unidade = df['car_prices_in_rupee'].str.strip().str.split(' ').str[-1]
unidade = unidade.where(unidade.isin(['Lakh', 'Crore']), 'sem unidade')
print(unidade.value_counts())
print(df.loc[unidade == 'sem unidade', 'car_prices_in_rupee'].head(3).tolist())

Se essa última categoria passar despercebida, `'99,999'` (99.999 rúpias) vira 99,999 lakh (quase 10 milhões) e contamina o treino. Olhar os valores distintos de cada coluna antes de convertê-la evita esse tipo de erro.

In [ ]:
def preco_em_rupias(texto):
    partes = texto.strip().split(' ')
    valor = float(partes[0].replace(',', ''))
    if partes[-1] == 'Lakh':
        return valor * 100_000
    if partes[-1] == 'Crore':
        return valor * 10_000_000
    return valor                                   # sem unidade: já está em rúpias

carros = pd.DataFrame({
    'preco': df['car_prices_in_rupee'].apply(preco_em_rupias),
    'ano': df['manufacture'],
    'km': df['kms_driven'].str.replace(' kms', '').str.replace(',', '').astype(float),
    'motor_cc': df['engine'].str.replace(' cc', '').astype(float),
    'lugares': df['Seats'].str.replace(' Seats', '').astype(float),
    'donos': df['ownership'].str[0].astype(float),
    'automatico': (df['transmission'] == 'Automatic').astype(float),
    'combustivel': df['fuel_type'],
    'marca': df['car_name'].str.split(' ').str[0],
})
carros.head()

In [ ]:
carros.describe().round(1)

O resto do nome (`'Compass 2.0 Longitude Option BSIV'`) fica de fora: são 1.896 versões diferentes, a maioria com um ou dois anúncios, poucos para a rede aprender algo sobre cada uma.

### Categorias: one-hot

`marca` (32 valores) e `combustivel` (5 valores) são categorias. Trocar cada marca por um número (Audi = 0, BMW = 1, ..., Volvo = 31, como faz o `LabelEncoder`) criaria uma ordem que não existe: para a rede, "Volvo" ficaria 31 vezes mais longe de "Audi" do que "BMW". Com **one-hot**, cada categoria vira uma coluna própria, com 1 quando o carro é daquela categoria e 0 nos outros casos. `pd.get_dummies` faz isso:

In [ ]:
carros = pd.get_dummies(carros, columns=['combustivel', 'marca'], dtype=float)
print(carros.shape)
carros.iloc[:3, 6:16]

### O alvo em escala logarítmica

Os preços variam de 35 mil a 19 milhões de rúpias, e a maioria dos carros fica abaixo de 1 milhão:

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5))
ax1.hist(carros['preco'] / 1e5, bins=60)
ax1.set_xlabel('preço (lakh = 100 mil rúpias)'); ax1.set_ylabel('nº de carros')
ax2.hist(np.log10(carros['preco']), bins=60)
ax2.set_xlabel('log10(preço em rúpias)')
plt.show()

Com o custo MSE sobre o preço em rúpias, um erro de 1 milhão num carro de 15 milhões pesaria tanto quanto o mesmo erro num carro de 300 mil: os carros caros dominariam o treino. Faz mais sentido medir o erro **relativo**: errar 10% para mais ou para menos, em qualquer faixa de preço. Treinar com $\log_{10}(\text{preço})$ faz isso, porque diferenças no log são razões no preço: $\log_{10}(1{,}1\,p) - \log_{10}(p) = \log_{10}(1{,}1) \approx 0{,}04$, para qualquer $p$. No histograma da direita, a distribuição fica bem mais simétrica.

Para voltar às rúpias, basta calcular $10^{\hat{y}}$.

### Treino, teste e padronização

Como no 5.5: 80% dos anúncios para treino, 20% para teste, scalers ajustados só no treino. Aqui cada linha é um anúncio diferente, então o sorteio por linhas está correto.

In [ ]:
X = carros.drop(columns='preco').values
y = np.log10(carros[['preco']].values)            # alvo: log10 do preço, shape (M, 1)
feature_names = carros.drop(columns='preco').columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_scaler = StandardScaler().fit(X_train)
y_scaler = StandardScaler().fit(y_train)
X_train_n, X_test_n = X_scaler.transform(X_train), X_scaler.transform(X_test)
y_train_n, y_test_n = y_scaler.transform(y_train), y_scaler.transform(y_test)

train_dataset = TensorDataset(torch.tensor(X_train_n, dtype=torch.float32), torch.tensor(y_train_n, dtype=torch.float32))
test_dataset = TensorDataset(torch.tensor(X_test_n, dtype=torch.float32), torch.tensor(y_test_n, dtype=torch.float32))
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000)

print('treino:', X_train_n.shape, '| teste:', X_test_n.shape)

## Modelo

A mesma rede do 5.5: duas camadas escondidas (64 e 32 neurônios) com ReLU e dropout, e uma saída sem ativação. Só o número de entradas muda: 6 features numéricas + 5 colunas de combustível + 32 de marca = 43.

In [ ]:
n_features = X_train_n.shape[1]

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(n_features, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)
        self.dropout1 = nn.Dropout(0.1)
        self.dropout2 = nn.Dropout(0.1)

    def forward(self, x):          # x: (N, 43)
        x = self.fc1(x)            # (N, 64)
        x = F.relu(x)
        x = self.dropout1(x)
        x = self.fc2(x)            # (N, 32)
        x = F.relu(x)
        x = self.dropout2(x)
        x = self.fc3(x)            # (N, 1): log10(preço) padronizado
        return x

model = Net()
print(model)
print('parâmetros:', sum(p.numel() for p in model.parameters()))

## Treinamento

Funções `train` e `test`, função de custo (`nn.MSELoss`) e agendador (`StepLR`, metade a cada 20 épocas) iguais aos do 5.5.

O otimizador também é o SGD com momentum, mas com uma taxa de aprendizado menor: `lr=0.003`. Com o `lr=0.01` do 5.5, o custo explode e vira `nan` nas primeiras épocas. Algumas features têm valores extremos (carros com quilometragem muito alta, por exemplo), que geram gradientes grandes; com uma taxa alta, um passo grande leva os pesos para longe, o próximo gradiente é ainda maior, e o treino diverge. Se o custo virar `nan` no seu trabalho, a primeira coisa a tentar é diminuir a taxa de aprendizado.

In [ ]:
def train(model, device, train_loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(data)
    return total_loss / len(train_loader.dataset)


def test(model, device, test_loader, criterion):
    model.eval()
    test_loss = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += criterion(output, target).item() * len(data)
    return test_loss / len(test_loader.dataset)

In [ ]:
model = Net().to(device)
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.003, momentum=0.9)
scheduler = StepLR(optimizer, step_size=20, gamma=0.5)

n_epochs = 60
train_losses, test_losses = [], []
for epoch in range(1, n_epochs + 1):
    train_loss = train(model, device, train_loader, optimizer, criterion)
    test_loss = test(model, device, test_loader, criterion)
    train_losses.append(train_loss)
    test_losses.append(test_loss)
    if epoch % 5 == 0 or epoch == 1:
        print(f'Época {epoch:2d} | lr = {scheduler.get_last_lr()[0]:.5f} | custo treino: {train_loss:.4f} | custo teste: {test_loss:.4f}')
    scheduler.step()

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(1, n_epochs + 1), train_losses, label='treino')
plt.plot(range(1, n_epochs + 1), test_losses, label='teste')
plt.xlabel('épocas'); plt.ylabel('custo (MSE no log do preço, padronizado)'); plt.yscale('log')
plt.legend(); plt.grid(True)
plt.show()

torch.save(model.state_dict(), "carros_mlp.pt")

## Avaliação

Para ler as previsões em rúpias, desfazemos os dois passos na ordem inversa: primeiro a padronização (`y_scaler.inverse_transform`), depois o log ($10^{\hat{y}}$).

Como o alvo foi pensado em termos relativos, a medida principal é o **erro percentual** $|\hat{p} - p| / p$ de cada carro, resumido pela mediana. Também mostramos o MAE em lakh. O baseline prevê, para todo carro, o preço mediano do treino.

In [ ]:
model.eval()
with torch.no_grad():
    pred_n = model(torch.tensor(X_test_n, dtype=torch.float32).to(device)).cpu().numpy()
pred = 10 ** y_scaler.inverse_transform(pred_n)    # rúpias
real = 10 ** y_test

erro_pct = np.abs(pred - real) / real * 100
base = np.median(10 ** y_train)
erro_pct_base = np.abs(base - real) / real * 100
print(f'rede:     erro percentual mediano = {np.median(erro_pct):.1f}% | MAE = {np.abs(pred - real).mean() / 1e5:.2f} lakh')
print(f'baseline: erro percentual mediano = {np.median(erro_pct_base):.1f}% | MAE = {np.abs(base - real).mean() / 1e5:.2f} lakh')
print(f'carros com erro abaixo de 20%: {(erro_pct < 20).mean() * 100:.0f}%')

In [ ]:
plt.figure(figsize=(5.5, 5.5))
plt.scatter(real / 1e5, pred / 1e5, s=8, alpha=0.5)
lims = [0.3, 200]
plt.plot(lims, lims, 'r--', label='previsão perfeita')
plt.xscale('log'); plt.yscale('log')
plt.xlabel('preço real (lakh)'); plt.ylabel('preço previsto (lakh)')
plt.title('Conjunto de teste')
plt.legend(); plt.grid(True, which='both', alpha=0.3)
plt.show()

Os eixos estão em escala log, em que distâncias iguais significam erros percentuais iguais. A nuvem acompanha a diagonal em toda a faixa, de carros de 50 mil a 10 milhões de rúpias. O erro que sobra vem em boa parte do que a tabela não registra: versão, estado de conservação e cidade do anúncio.

## Exercícios

1. Estime o preço de um carro que não está nos dados: um Hyundai, ano 2018, 45.000 km, motor de 1.497 cc, 5 lugares, primeiro dono, câmbio manual, a gasolina (`'Petrol'`). Monte uma linha com as **mesmas colunas, na mesma ordem** de `feature_names` (comece com zeros e preencha), aplique `X_scaler`, passe pela rede e desfaça a padronização e o log. Depois troque a marca para BMW e o câmbio para automático e compare.

In [ ]:
def prever_preco(ano, km, motor_cc, lugares, donos, automatico, combustivel, marca):
    # TODO: escreva sua solução aqui

p1 = prever_preco(2018, 45000, 1497, 5, 1, 0, 'Petrol', 'Hyundai')
p2 = prever_preco(2018, 45000, 1497, 5, 1, 1, 'Petrol', 'BMW')
print(f'Hyundai manual:  {p1 / 1e5:.2f} lakh')
print(f'BMW automático:  {p2 / 1e5:.2f} lakh')

*Escreva sua resposta aqui.*

2. Treine uma rede nova com o preço em rúpias, **sem o log** (padronize `10 ** y_train` direto com um `StandardScaler`), pelo mesmo número de épocas, e calcule o erro percentual mediano e o MAE em lakh no teste, no total e só para os carros abaixo de 5 lakh. Antes de executar, preveja em que faixa de preço a rede sem log vai errar mais, em termos percentuais. Por quê?

In [ ]:
# TODO: escreva sua solução aqui
print(f'com log: erro percentual mediano = {np.median(erro_pct):.1f}% | MAE = {np.abs(pred - real).mean() / 1e5:.2f} lakh')
print(f'sem log: erro percentual mediano = {np.median(erro_pct_lin):.1f}% | MAE = {mae_lin:.2f} lakh')
baratos = real < 5e5                               # carros abaixo de 5 lakh
print(f'carros abaixo de 5 lakh ({baratos.sum()}): erro mediano com log = {np.median(erro_pct[baratos]):.1f}% | sem log = {np.median(erro_pct_lin[baratos]):.1f}%')

*Escreva sua resposta aqui.*

## Aprendizados

- Em dados reais, boa parte do trabalho é transformar texto em números. Olhe os valores distintos de cada coluna antes de convertê-la: formatos inesperados (preços sem unidade) geram erros silenciosos.
- Categorias entram na rede como colunas one-hot (`pd.get_dummies`), não como números arbitrários, que criariam uma ordem inexistente.
- Quando o alvo varia em várias ordens de grandeza e o erro relativo é o que importa, treine com o log do alvo e volte com $10^{\hat{y}}$ na avaliação.
- Para usar o modelo num dado novo, aplique a mesma preparação do treino (mesmas colunas, mesma ordem, os mesmos scalers) e desfaça as transformações do alvo.